# nn.Module: Building Neural Block by Block

Reach for this when you need: 
- To define custom layers or full architectures.
- Reference for `parameters`, `buffers`, and `state_dict` management.
- To understand hooks and model sub-modules.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Defining a Custom Module

### `torch.nn.Module` Essentials

| Component | Description | Usage |
| :--- | :--- | :--- |
| `__init__` | Define layers (Linear, Conv, etc.) | Logic for param initialization |
| `forward` | Define the data flow | Logic for compute graph |
| `parameters` | Generator for all learnable weights | Needed by optimizers |
| `buffers` | Tensors that don't need grad (e.g. running mean) | Persistent state during save/load |

In [2]:
class SimpleMLP(nn.Module):
    """
    A basic Multi-Layer Perceptron as a reference for nn.Module structure.
    Parameters:
    - input_dim (int): Number of input features
    - hidden_dim (int): Hidden layer size
    - output_dim (int): Output class count
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleMLP(input_dim=10, hidden_dim=20, output_dim=2).to(device)

## 2. Parameter and Buffer Management

**torch.nn.Parameter**
A subclass of Tensor that is automatically registered as a model parameter.

✅ **Use when**: Creating custom learnable weights not covered by built-in layers.
❌ **Don't use when**: Standard layers like `nn.Linear` already exist.

In [4]:
class CustomLayer(nn.Module):
    def __init__(self, size: int):
        super().__init__()
        # Automatically added to model.parameters()
        self.weight = nn.Parameter(torch.randn(size))
        
        # Registered as a buffer (saved in state_dict, but NOT optimized)
        self.register_buffer("running_stat", torch.zeros(1))
 
    def forward(self, x):
        return x * self.weight

## 3. Containers

| Container | Usage |
| :--- | :--- |
| `nn.Sequential` | Static pipe of layers; no control flow possible in `forward` |
| `nn.ModuleList` | List of modules; registers parameters properly. Use with for-loops |
| `nn.ModuleDict` | Key-value store for modules; useful for switching between heads |

In [6]:
layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(5)])

def forward(self, x):
    for layer in layers:
        x = F.relu(layer(x))
    return x

### Common Pitfalls
- **Forgotten `super().__init__()`**: This will cause `AttributeError` when accessing layers.
- **F.functional vs nn.Module**: Use `nn.ReLU` in `__init__` if you need state (none for ReLU), but generally prefer `nn.Module` for layers with weights and `F` for stateless ops (dropout, pool, activations).
- **Moving to device**: Calling `model.to(device)` is recursive. It moves ALL sub-modules and parameters.

### Key Takeaways
- `nn.Module` is the base class for everything from a single layer to a GPT-4 architecture.
- Always register custom tensors as `nn.Parameter` if they need gradients.
- Use `state_dict()` for saving/loading model weights only (no architecture).